In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
SELECTED_FOLDER = '07.2026'

In [ ]:
def load_data(file_path: str) -> pd.DataFrame:
    frames = []
    try:
        for files in Path(file_path).glob('*.csv'):
            df = pd.read_csv(files)
            report_date = pd.to_datetime(files.stem, format='%Y.%m.%d')
            df['report_date'] = report_date
            frames.append(df)
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=False).sort_values(by='report_date', ignore_index=True)

In [ ]:
source_dir = Path.cwd().parent / 'data' / 'raw' / SELECTED_FOLDER

In [ ]:
sales_raw = load_data(source_dir)

In [ ]:
sales_raw

In [ ]:
sales_raw = sales_raw.drop(['Okres', 'Location'], axis=1)

In [ ]:
sales_raw

In [ ]:
sales_raw = sales_raw.rename({
    'Typ': 'product_type',
    'Nazwa przedmiotu': 'product_name',
    'Sprzedany': 'total_sales',
    'Cena': 'total_revenue',
    'report_date': 'date',
}, axis=1)

In [ ]:
sales_raw

In [ ]:
sales = sales_raw.dropna()

In [ ]:
sales

In [ ]:
sales.columns[sales.isna().any()].to_list() # There are no NaN columns.

In [ ]:
sales = sales.drop('product_type', axis=1)

In [ ]:
sales = sales[sales['product_name'] != '12 szt'] # Trash data

In [ ]:
sorted(sales['product_name'].unique().tolist())

In [ ]:
product_names = pd.DataFrame({'product_name_from_report': sorted(sales['product_name'].unique().tolist())})

In [ ]:
product_names

In [ ]:
my_calculations = pd.read_csv(Path.cwd().parent / 'data' / 'mappings' / '07.2026' / 'my_calculations.csv')

In [ ]:
my_calculations = my_calculations[['ПОЗИЦИЯ', 'РЕСТ', 'КАТЕГОРИЯ', 'РАСХОДЫ', 'VAT ПРОЦ']]

In [ ]:
my_calculations = my_calculations[my_calculations['РЕСТ'] == 'Vibe'].sort_values(by='ПОЗИЦИЯ')

In [ ]:
my_calculations

In [ ]:
my_calculations = my_calculations.drop('РЕСТ', axis=1)

In [ ]:
my_calculations = my_calculations.rename({
    'ПОЗИЦИЯ': 'name_from_my_calculations',
    'КАТЕГОРИЯ': 'product_category_from_my_calculations',
    'РАСХОДЫ': 'product_cost_my_calculations',
    'VAT ПРОЦ': 'vat_rate_my_calculations',
}, axis=1)

In [ ]:
my_calculations = my_calculations.sort_values(by='name_from_my_calculations', ignore_index=True)

In [ ]:
my_calculations

In [ ]:
product_costs_mapping = pd.DataFrame(
    data={
        'product_name_from_report': product_names['product_name_from_report'],
        'product_name_from_calculations_accepted': '',
        'product_category_accepted': '',
        'vat_rate_accepted': '',
        'product_cost_accepted': '',
        'name_from_my_calculations': my_calculations['name_from_my_calculations'],
        'product_category_from_my_calculations': my_calculations['product_category_from_my_calculations'],
        'product_cost_my_calculations': my_calculations['product_cost_my_calculations'],
        'vat_rate_my_calculations': my_calculations['vat_rate_my_calculations'],
    }
)

In [ ]:
product_costs_mapping

In [ ]:
# product_costs_mapping.to_csv(Path.cwd().parent / 'data' / 'mappings' / '07.2026' /'product_costs_mapping.csv', index=False)

In [ ]:
costs_mapping = pd.read_csv(Path().cwd().parent / 'data' / 'mappings' / '07.2026' / 'product_costs_mapping.csv')

In [ ]:
costs_mapping

In [ ]:
costs_table = costs_mapping[['product_name_from_report', 'product_name_from_calculations_accepted', 'product_name_unified', 'product_category_accepted', 'product_cost_accepted', 'vat_rate_accepted']]

In [ ]:
costs_table

In [ ]:
# Keep the original report name as the join key.
costs_table = costs_table.drop('product_name_from_calculations_accepted', axis=1).rename(columns={
    'product_name_from_report': 'product_name',
    'product_category_accepted': 'product_category',
    'product_cost_accepted': 'product_cost',
    'vat_rate_accepted': 'vat_rate'
})

In [ ]:
costs_table

In [ ]:
sales_final = sales.merge(
    right=costs_table,
    how='left',
    on='product_name',
    validate='many_to_one'
)

In [ ]:
sales_final

In [ ]:
sales_grouby_revenue = sales_final.groupby(by='product_name')[['total_sales', 'total_revenue']].sum().sort_values(by='total_revenue', ascending=False)

In [ ]:
sales_grouby_revenue

In [ ]:
sales_grouby_revenue['total_revenue'].sum()

In [ ]:
sales_final['product_name'].to_list()

In [ ]:
sales_final

### Metrics formulas

In [ ]:
sales_metrics = sales_final

In [ ]:
sales_metrics

In [ ]:
sales_metrics['unit_price_gross'] = sales_final['total_revenue'] / sales_final['total_sales']

In [ ]:
sales_metrics['unit_vat'] = sales_final['unit_price_gross'] * sales_final['vat_rate'] / (100 + sales_final['vat_rate'])

In [ ]:
sales_metrics['unit_price_net'] = sales_final['unit_price_gross'] - sales_final['unit_vat']

In [ ]:
sales_metrics['unit_profit_est'] = sales_final['unit_price_net'] - sales_final['product_cost']
sales_metrics['margin_rate'] = sales_final['unit_profit_est'] / sales_final['unit_price_net']
sales_metrics['food_cost_rate'] = sales_final['product_cost'] / sales_final['unit_price_net']
sales_metrics['total_revenue_net'] = sales_final['total_sales'] * sales_final['unit_price_net']
sales_metrics['total_revenue_gross'] = sales_final['total_sales'] * sales_final['unit_price_gross']
sales_metrics['total_cost_est'] = sales_final['total_sales'] * sales_final['product_cost']
sales_metrics['total_profit_est'] = sales_final['total_revenue_net'] - sales_final['total_cost_est']

In [ ]:
sales_metrics

In [ ]:
sales_metrics = sales_metrics[['date', 'product_name', 'product_name_unified', 'product_category', 'total_sales', 'total_revenue', 'product_cost', 'vat_rate', 'unit_price_gross', 'unit_vat', 'unit_price_net',
                              'unit_profit_est', 'margin_rate', 'food_cost_rate', 'total_revenue_net', 'total_revenue_gross', 'total_cost_est', 'total_profit_est']]

In [ ]:
sales_metrics

In [401]:
sales_metrics = sales_metrics.drop('product_name', axis=1).rename({'product_name_unified': 'product_name'}, axis=1)

In [403]:
sales_metrics.dropna()

,date,product_name,product_category,total_sales,total_revenue,product_cost,vat_rate,unit_price_gross,unit_vat,unit_price_net,unit_profit_est,margin_rate,food_cost_rate,total_revenue_net,total_revenue_gross,total_cost_est,total_profit_est
0,2026-07-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,FUTOMAKI,12,456.0,7.51,8.0,38.0,2.814815,35.185185,27.675185,0.786558,0.213442,422.222222,456.0,90.12,332.102222
1,2026-07-01,Wasabi,DODATKI,1,2.0,0.30,8.0,2.0,0.148148,1.851852,1.551852,0.838000,0.162000,1.851852,2.0,0.30,1.551852
2,2026-07-01,Krabs - Burger,SUSHI BURGER,1,25.0,12.27,8.0,25.0,1.851852,23.148148,10.878148,0.469936,0.530064,23.148148,25.0,12.27,10.878148
3,2026-07-01,Sos teriyaki 30ml,DODATKI,1,3.0,1.00,8.0,3.0,0.222222,2.777778,1.777778,0.640000,0.360000,2.777778,3.0,1.00,1.777778
4,2026-07-01,Sos sojowy 30ml,DODATKI,2,6.0,0.50,8.0,3.0,0.222222,2.777778,2.277778,0.820000,0.180000,5.555556,6.0,1.00,4.555556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797,2026-07-31,Futomaki KREWETKA PANKO | 6 szt,FUTOMAKI,1,40.0,8.72,23.0,40.0,7.479675,32.520325,23.800325,0.731860,0.268140,32.520325,40.0,8.72,23.800325
798,2026-07-31,Zapieczona rolka ŁOSOŚ SUROWY | 8 szt,ROLKI ZAPIECZONE,1,60.0,12.11,8.0,60.0,4.444444,55.555556,43.445556,0.782020,0.217980,55.555556,60.0,12.11,43.445556
799,2026-07-31,Fuji roll | 8 szt,ROLKI AUTORSKIE,1,65.0,14.86,8.0,65.0,4.814815,60.185185,45.325185,0.753095,0.246905,60.185185,65.0,14.86,45.325185
800,2026-07-31,Zapieczona rolka FELIX | 8 szt,ROLKI ZAPIECZONE,1,60.0,10.83,8.0,60.0,4.444444,55.555556,44.725556,0.805060,0.194940,55.555556,60.0,10.83,44.725556


In [404]:
sales_metrics.to_csv(Path().cwd().parent / 'data' / 'output' / 'sales_07_2026.csv')